# Ground-Station Passes with Three Antenna Types

Run the same day in LEO with a **fixed**, **omni**, and **gimballed** antenna.
`PassTimes` books ground-station passes before execution for every antenna type.
The antenna determines whether using a pass requires spacecraft tracking:

| Antenna | `Pass.requires_tracking` | Operation during a scheduled pass |
| --- | --- | --- |
| Fixed | `True` | Slew to the station, track in `PASS` mode, pause science |
| Omni | `False` | Downlink while science continues |
| Gimballed | `False` | Downlink while science continues, subject to reach and repoint outages |

All three runs use the same orbit, targets, station scheduling probabilities,
nominal data rates, and random seed. The main comparison uses a full-range
gimbal to isolate repoint outages. A final run limits its reach.

Run all cells with a Python environment containing this checkout and its
notebook dependencies (`pip install -e ".[jupyter]"` from the repository root).
Start Jupyter in the repository root or `examples/`. The bundled TLE fixes
this example's orbit and date; use a mission-specific ephemeris for other dates.

In [ ]:
from datetime import datetime, timedelta, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from rust_ephem import EarthLimbConstraint, SunConstraint, TLEEphemeris

from conops import DITL
from conops.common import ACSMode, AntennaType, ObsType
from conops.config import (
    AntennaPointing,
    AttitudeControlSystem,
    BandCapability,
    Battery,
    CommunicationsSystem,
    Constraint,
    DataGeneration,
    GroundStationRegistry,
    HousekeepingData,
    Instrument,
    MissionConfig,
    OnboardRecorder,
    Payload,
    RadiatorConfiguration,
    ScienceData,
    SolarPanelSet,
    SpacecraftBus,
    StarTrackerConfiguration,
)
from conops.schedulers import PriorityScheduler
from conops.simulation import PassTimes
from conops.targets import Plan, Pointing

tle_path = next(
    (p for p in (Path("example.tle"), Path("examples/example.tle")) if p.is_file()),
    None,
)
if tle_path is None:
    raise FileNotFoundError("Run from the repository root or examples/.")

BEGIN = datetime(2025, 11, 1, tzinfo=timezone.utc)
END = BEGIN + timedelta(days=1)
STEP_SECONDS = 60
SEED = 1234
RATE_MBPS = 10.0
CAPACITY_GB = 128.0
IMAGER = "Imager"
ephemeris = TLEEphemeris(
    tle=str(tle_path), begin=BEGIN, end=END, step_size=STEP_SECONDS
)

## 1. Configure the spacecraft

Power and hardware exclusions are simplified so this comparison focuses on
communications. Sun and Earth-limb constraints still apply. The imager produces
1 Mbps during science; housekeeping produces 0.1 Mbps continuously.

Each run gets its own configuration, recorder, and targets. The recorder starts
with 16 Gb so early passes have data to send; its capacity keeps storage from
dominating this example. Rates and recorder volumes use **bits**: Mbps and Gb.

A station has a 65% chance of accepting each eligible pass. Scheduling also
applies the eight-minute minimum and selects non-overlapping passes.
Fixed passes additionally need safe spacecraft tracking profiles.

In [ ]:
def make_config(antenna_type, *, gimbal_range_deg=180.0, probability=0.65):
    stations = GroundStationRegistry.default()
    for station in stations.stations:
        station.schedule_probability = probability

    pointing = {"antenna_type": antenna_type}
    if antenna_type == AntennaType.GIMBALED:
        pointing.update(
            gimbal_range_deg=gimbal_range_deg,
            repoint_threshold_deg=10.0,
            repoint_seconds=20.0,
        )

    config = MissionConfig(
        name=f"{antenna_type.value} antenna comparison",
        random_seed=SEED,
        ground_stations=stations,
        recorder=OnboardRecorder(capacity_gb=CAPACITY_GB, current_volume_gb=16.0),
        constraint=Constraint(
            sun_constraint=SunConstraint(min_angle=45),
            earth_constraint=EarthLimbConstraint(min_angle=10),
        ),
        battery=Battery(watthour=100_000.0),
        solar_panel=SolarPanelSet(panels=[]),
        spacecraft_bus=SpacecraftBus(
            communications=CommunicationsSystem(
                name="S-band",
                band_capabilities=[
                    BandCapability(band="S", downlink_rate_mbps=RATE_MBPS)
                ],
                antenna_pointing=AntennaPointing(**pointing),
            ),
            attitude_control=AttitudeControlSystem(
                max_slew_rate=1.0, slew_acceleration=0.5, settle_time=30.0
            ),
            data_generation=HousekeepingData(rate_gbps=0.0001),
            star_trackers=StarTrackerConfiguration(
                star_trackers=[], min_functional_trackers=0, modes_require_lock=[]
            ),
            radiators=RadiatorConfiguration(radiators=[]),
        ),
        payload=Payload(
            instruments=[
                Instrument(
                    name=IMAGER,
                    data_generation=DataGeneration(
                        science=ScienceData(rate_gbps=0.001)
                    ),
                )
            ]
        ),
    )
    config.constraint.ephem = ephemeris
    return config


def make_targets(config):
    targets = []
    for i, (ra, dec) in enumerate(
        [(105, 10), (110, 10), (115, 10), (105, 0), (110, 20)]
    ):
        target = Pointing(
            config=config,
            instrument_name=IMAGER,
            obsid=100 + i,
            name=f"Target {i + 1}",
            ra=ra,
            dec=dec,
            merit=90 - i,
            ss_min=5 * 60,
            ss_max=20 * 60,
        )
        target.exptime = 24 * 3600
        targets.append(target)
    return targets


antenna_types = {
    "Fixed": AntennaType.FIXED,
    "Omni": AntennaType.OMNI,
    "Gimballed": AntennaType.GIMBALED,
}
configs = {label: make_config(kind) for label, kind in antenna_types.items()}

## 2. Book the day's passes before planning science

Use the same `PassTimes.get(year, day, length)` API for every antenna. No
simulation has started yet. A common seed makes booking draws reproducible;
spacecraft tracking constraints can still affect which passes are feasible.

`requires_tracking` describes the spacecraft activity needed to use a pass.
The schedule remains fixed during execution and science replanning.

In [ ]:
bookings = {}
schedule_rows = []
for label, config in configs.items():
    passes = PassTimes(config)
    passes.get(BEGIN.year, BEGIN.timetuple().tm_yday, length=1)
    bookings[label] = passes
    for contact in passes.passes:
        schedule_rows.append(
            {
                "Antenna": label,
                "Station": contact.station,
                "Start UTC": datetime.fromtimestamp(
                    contact.begin, timezone.utc
                ).strftime("%H:%M"),
                "Duration (min)": contact.length / 60,
                "Requires tracking": contact.requires_tracking,
            }
        )

schedule = pd.DataFrame(schedule_rows)
display(schedule.sort_values(["Start UTC", "Antenna"]).reset_index(drop=True))
assert all(bookings[label].passes for label in antenna_types)
assert all(p.requires_tracking for p in bookings["Fixed"].passes)
assert all(
    not p.requires_tracking
    for label in ("Omni", "Gimballed")
    for p in bookings[label].passes
)
assert [(p.begin, p.end, p.station) for p in bookings["Omni"].passes] == [
    (p.begin, p.end, p.station) for p in bookings["Gimballed"].passes
]

## 3. Plan and execute against those bookings

Supply the booked passes to `PriorityScheduler`. It reserves spacecraft
tracking activities only where `requires_tracking=True`. Link coverage accounts
for the usable portions of the other passes, including gimbal repoint outages.

Every plan has more science requested than one day can collect. The executed
plan should match the planned science and remain within the recorder limit.

In [ ]:
def execute_day(config, passes):
    scheduler = PriorityScheduler(
        config,
        make_targets(config),
        BEGIN,
        END,
        step_size=STEP_SECONDS,
        passes=passes.passes,
    )
    plan = scheduler.schedule()
    run = DITL(config=config, plan=plan, begin=BEGIN, end=END)
    run.step_size = STEP_SECONDS
    assert run.calc()
    assert run.validate_plan_matches_execution() == []
    assert max(run.recorder_volume_gb) <= config.recorder.planning_limit_gb + 1e-6
    return run


runs = {label: execute_day(configs[label], bookings[label]) for label in configs}


def summarize(label, run, passes):
    increments = np.diff(np.r_[0.0, run.data_downlinked_gb])
    science_steps = np.array(run.mode) == ACSMode.SCIENCE
    return {
        "Antenna": label,
        "Booked passes": len(passes.passes),
        "Tracking activities": sum(e.obstype == ObsType.GSP for e in run.plan),
        "Science (h)": sum(
            record.collection_seconds or 0.0 for record in run.telemetry.housekeeping
        )
        / 3600,
        "PASS mode (min)": sum(mode == ACSMode.PASS for mode in run.mode)
        * STEP_SECONDS
        / 60,
        "Downlinked (Gb)": run.data_downlinked_gb[-1],
        "During science (Gb)": increments[science_steps].sum(),
        "Final recorder (Gb)": run.recorder_volume_gb[-1],
        "Tracking repoints": sum(
            repoint.kind == "tracking"
            for repoint in run.coverage.repoints_in(BEGIN.timestamp(), END.timestamp())
        )
        if run.coverage is not None
        else 0,
    }


comparison = pd.DataFrame(
    [summarize(label, run, bookings[label]) for label, run in runs.items()]
).set_index("Antenna")
display(comparison.round(3).T)

assert comparison.loc["Fixed", "Tracking activities"] > 0
assert comparison.loc["Fixed", "PASS mode (min)"] > 0
assert comparison.loc["Fixed", "During science (Gb)"] == 0
for label in ("Omni", "Gimballed"):
    assert comparison.loc[label, "Tracking activities"] == 0
    assert comparison.loc[label, "PASS mode (min)"] == 0
    assert comparison.loc[label, "During science (Gb)"] > 0
assert comparison.loc["Gimballed", "Tracking repoints"] > 0

## 4. Compare the timelines and data flow

Blue bars show the booked passes. Green bars mark telemetry steps collecting
science, and purple bars mark steps transferring data. A step is one minute;
shorter setup times and outages are accounted for in the summary above.
Repoint markers show when the gimballed antenna moves.

Look for science stopping during fixed-antenna tracking, and science and
downlink occurring together in the other runs. Gaps between scheduled passes
remain gaps in ground contact for every antenna type.

Overlapping curves are expected: fixed and omni runs can downlink the same
amount through identical passes. The omni and full-range gimbal collect the
same science here because repoints are allowed during observations; gimbal
repoint outages reduce the data transferred.

In [ ]:
def step_spans(run, mask):
    times = np.asarray(run.utime)
    durations = np.minimum(STEP_SECONDS, END.timestamp() - times)
    return [
        ((time - BEGIN.timestamp()) / 3600, duration / 3600)
        for time, duration, selected in zip(times, durations, mask)
        if selected
    ]


fig, axes = plt.subplots(3, 1, figsize=(13, 6.5), sharex=True)
for ax, (label, run) in zip(axes, runs.items()):
    booked_spans = [
        (
            (p.begin - BEGIN.timestamp()) / 3600,
            (min(p.end, END.timestamp()) - p.begin) / 3600,
        )
        for p in bookings[label].passes
    ]
    science = [
        (record.collection_seconds or 0.0) > 0 for record in run.telemetry.housekeeping
    ]
    transferring = np.diff(np.r_[0.0, run.data_downlinked_gb]) > 1e-10
    ax.broken_barh(booked_spans, (1.8, 0.5), facecolors="#3467a3")
    ax.broken_barh(step_spans(run, transferring), (0.9, 0.5), facecolors="#8052a1")
    ax.broken_barh(step_spans(run, science), (0.0, 0.5), facecolors="#238b62")
    if run.coverage is not None:
        for repoint in run.coverage.repoints_in(BEGIN.timestamp(), END.timestamp()):
            if repoint.kind == "tracking":
                ax.vlines(
                    (repoint.time - BEGIN.timestamp()) / 3600,
                    0.85,
                    1.45,
                    color="#cb4a32",
                    linewidth=1,
                )
    ax.set_yticks([0.25, 1.15, 2.05], ["Science", "Data transfer", "Booked pass"])
    ax.set_title(label, loc="left", fontsize=11, fontweight="bold")
    ax.set_ylim(-0.15, 2.5)
    ax.grid(axis="x", alpha=0.2)
axes[-1].set(xlim=(0, 24), xlabel="Hours since 00:00 UTC")
fig.suptitle("Scheduled ground-station passes and spacecraft activity", fontsize=14)
fig.tight_layout()
plt.show()

colors = {"Fixed": "#3467a3", "Omni": "#238b62", "Gimballed": "#8052a1"}
line_styles = {"Fixed": "-", "Omni": "--", "Gimballed": ":"}
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
for label, run in runs.items():
    hours = (np.asarray(run.utime) + STEP_SECONDS - BEGIN.timestamp()) / 3600
    science_hours = (
        np.cumsum(
            [record.collection_seconds or 0.0 for record in run.telemetry.housekeeping]
        )
        / 3600
    )
    style = {"label": label, "color": colors[label], "linestyle": line_styles[label]}
    axes[0].plot(hours, science_hours, **style)
    axes[1].plot(hours, run.data_downlinked_gb, **style)
    axes[2].plot(hours, run.recorder_volume_gb, **style)
for ax, ylabel in zip(
    axes, ["Science collected (h)", "Downlinked (Gb)", "Recorder (Gb)"]
):
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.2)
axes[0].legend(ncol=3, frameon=False)
axes[-1].set(xlim=(0, 24), xlabel="Hours since 00:00 UTC")
fig.suptitle("Science and data flow over the day", fontsize=14)
fig.tight_layout()
plt.show()

## 5. Limit the gimbal's reach

The main gimbal has an idealized 180-degree range from its mounting boresight.
Here, reduce that range to 60 degrees while keeping the same booked passes.
A booking reserves ground-station time; it does not guarantee that the flown
spacecraft attitude lets the antenna reach the station.

Planning currently assumes a visible booked station is within gimbal reach.
Execution checks reach against the flown attitude, so actual downlink can be
lower than the recorder forecast. The large recorder gives this example margin.
Repoint times are calculated from station geometry and are not reset by
spacecraft slews. The default observation policy allows repoints during science;
use an observation category with `antenna_repoints="forbidden"` when jitter matters.

In [ ]:
restricted_config = make_config(AntennaType.GIMBALED, gimbal_range_deg=60.0)
restricted_passes = PassTimes(restricted_config)
restricted_passes.restore_bookings(
    [(p.begin, p.end, p.station) for p in bookings["Gimballed"].passes]
)
restricted_run = execute_day(restricted_config, restricted_passes)
reach_comparison = pd.DataFrame(
    [
        summarize("Gimbal 180 degrees", runs["Gimballed"], bookings["Gimballed"]),
        summarize("Gimbal 60 degrees", restricted_run, restricted_passes),
    ]
).set_index("Antenna")
display(
    reach_comparison[
        ["Booked passes", "Science (h)", "Downlinked (Gb)", "Final recorder (Gb)"]
    ]
    .round(3)
    .T
)
assert restricted_run.data_downlinked_gb[-1] < runs["Gimballed"].data_downlinked_gb[-1]

## 6. Booking probability and replay

Setting every station's scheduling probability to zero books no passes for
any antenna. Saved bookings are preserved with the plan; replay uses them
without drawing scheduling probabilities again.

In [ ]:
for label, kind in antenna_types.items():
    unavailable = PassTimes(make_config(kind, probability=0.0))
    unavailable.get(BEGIN.year, BEGIN.timetuple().tm_yday, length=1)
    assert unavailable.passes == []
    print(f"{label}: zero probability -> {len(unavailable.passes)} booked passes")

# Round-trip through the saved-plan JSON format without writing a file.
saved_plan = Plan.model_validate_json(
    runs["Omni"].plan.model_dump_json(exclude_none=True)
)
replay = DITL(
    config=make_config(AntennaType.OMNI, probability=0.0),
    plan=saved_plan,
    begin=BEGIN,
    end=END,
)
assert replay.plan.ground_contacts == runs["Omni"].plan.ground_contacts
assert replay.coverage.contacts == runs["Omni"].coverage.contacts
print(f"Replay preserved {len(replay.plan.ground_contacts)} booked passes.")

All antenna types use scheduled ground-station passes. Spacecraft tracking,
antenna reach, and repoint outages determine how much of each booked pass is
usable and whether science can continue.

See the communications, planning, and data-management guides for the model and
configuration fields. To explore station access, change `probability`; to
explore tracking outages, change `repoint_threshold_deg` and `repoint_seconds`.